In [1]:
import pandas as pd
import sqlite3
import os

DATA_FOLDER = r"C:\Users\LENOVO\Downloads\Saas_Churn_Project\data"
conn = sqlite3.connect('saas_churn_analysis.db')   # Database name you want

files = {
    "accounts": "ravenstack_accounts.csv",
    "subscriptions": "ravenstack_subscriptions.csv",
    "usage": "ravenstack_feature_usage.csv",
    "tickets": "ravenstack_support_tickets.csv",
    "churn": "ravenstack_churn_events.csv"
}

for name, file in files.items():
    path = os.path.join(DATA_FOLDER, file)
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.to_sql(name, conn, if_exists='replace', index=False)



In [2]:
print("===: DATA CLEANING & QUALITY CHECK ===\n")

# 1. Check how many rows in each table
print("1. Number of rows in each table:")
print(pd.read_sql("SELECT 'accounts' AS table_name, COUNT(*) AS rows FROM accounts", conn))
print(pd.read_sql("SELECT 'subscriptions' AS table_name, COUNT(*) AS rows FROM subscriptions", conn))
print(pd.read_sql("SELECT 'usage' AS table_name, COUNT(*) AS rows FROM usage", conn))
print(pd.read_sql("SELECT 'tickets' AS table_name, COUNT(*) AS rows FROM tickets", conn))
print(pd.read_sql("SELECT 'churn' AS table_name, COUNT(*) AS rows FROM churn", conn))

# 2. Check for missing values
print("\n2. Missing Values Check:")
print(pd.read_sql("SELECT SUM(CASE WHEN industry IS NULL THEN 1 ELSE 0 END) AS missing_industry FROM accounts", conn))
print(pd.read_sql("SELECT SUM(CASE WHEN plan_tier IS NULL THEN 1 ELSE 0 END) AS missing_plan_tier FROM subscriptions", conn))
print(pd.read_sql("SELECT SUM(CASE WHEN feedback_text IS NULL THEN 1 ELSE 0 END) AS missing_feedback FROM churn", conn))

print("\n Data Cleaning Complete!")

===: DATA CLEANING & QUALITY CHECK ===

1. Number of rows in each table:
  table_name  rows
0   accounts   500
      table_name  rows
0  subscriptions  5000
  table_name   rows
0      usage  25000
  table_name  rows
0    tickets  2000
  table_name  rows
0      churn   600

2. Missing Values Check:
   missing_industry
0                 0
   missing_plan_tier
0                  0
   missing_feedback
0               148

 Data Cleaning Complete!


In [3]:

# ================== ] MASTER TABLE (With All Columns) ==================

conn.execute("DROP TABLE IF EXISTS master")

master_query = """
CREATE TABLE master AS
WITH 
    ticket_count AS (SELECT account_id, COUNT(*) AS tickets FROM tickets GROUP BY account_id),
    avg_usage AS (SELECT subscription_id, ROUND(AVG(usage_count), 1) AS usage FROM usage GROUP BY subscription_id)
SELECT 
    a.account_id,
    a.industry,
    s.plan_tier,
    s.start_date,
    s.end_date,
    s.mrr_amount,
    c.feedback_text,                    -- ← Added back
    CASE WHEN s.end_date IS NOT NULL THEN 1 ELSE 0 END AS churn_flag,
    ROUND((JULIANDAY(COALESCE(s.end_date, DATE('now'))) - JULIANDAY(s.start_date)) / 30.437, 1) AS tenure_months,
    COALESCE(t.tickets, 0) AS ticket_count,
    COALESCE(u.usage, 0) AS avg_usage,
    strftime('%Y-%m', s.start_date) AS signup_month
FROM accounts a
LEFT JOIN subscriptions s ON a.account_id = s.account_id
LEFT JOIN churn c ON a.account_id = c.account_id
LEFT JOIN ticket_count t ON a.account_id = t.account_id
LEFT JOIN avg_usage u ON s.subscription_id = u.subscription_id;
"""

conn.execute(master_query)
print(" Master table created successfully with all required columns!")

 Master table created successfully with all required columns!


In [9]:
print(" KEY BUSINESS METRICS")
print(pd.read_sql("""
    SELECT 
        COUNT(*) AS total_subscriptions,
        ROUND(AVG(churn_flag)*100, 1) AS churn_rate,
        ROUND(SUM(CASE WHEN churn_flag = 0 THEN mrr_amount ELSE 0 END), 0) AS active_mrr,
        ROUND(SUM(CASE WHEN churn_flag = 1 THEN mrr_amount ELSE 0 END)*12, 0) AS arr_at_risk
    FROM master
""", conn))

 KEY BUSINESS METRICS
   total_subscriptions  churn_rate  active_mrr  arr_at_risk
0                 7429         9.9  14841284.0   20377668.0


In [5]:
print("Churn by Plan Tier:")
print(pd.read_sql("SELECT plan_tier, COUNT(*) as subs, ROUND(AVG(churn_flag)*100,1) as churn_rate FROM master GROUP BY plan_tier ORDER BY churn_rate DESC", conn))



Churn by Plan Tier:
    plan_tier  subs  churn_rate
0         Pro  2467        10.3
1  Enterprise  2551         9.9
2       Basic  2411         9.5


In [6]:
from collections import Counter  

print(" WHAT CUSTOMERS ARE SAYING - Churn Feedback Analysis")

feedback = pd.read_sql("""
    SELECT feedback_text 
    FROM master 
    WHERE churn_flag = 1 AND feedback_text IS NOT NULL
""", conn)

if len(feedback) > 0:
    all_text = ' '.join(feedback['feedback_text'].astype(str).tolist()).lower()
    words = [word for word in all_text.split() if len(word) >= 3]

    common_words = ['the','and','for','are','but','not','you','can','was','our','get','has','how','its','now','did','this','that','with','they','from','have','been','more','will','what','when','your','than','some','just','like','make','need','want']
    clean_words = [w for w in words if w not in common_words]

    top_words = Counter(clean_words).most_common(12)

    print("Top reasons customers are leaving:\n")
    for rank, (word, count) in enumerate(top_words, 1):
        print(f"{rank:2d}. {word:<15} ({count} times)")
else:
    print("No feedback text available.")

 WHAT CUSTOMERS ARE SAYING - Churn Feedback Analysis
Top reasons customers are leaving:

 1. too             (171 times)
 2. expensive       (171 times)
 3. switched        (145 times)
 4. competitor      (145 times)
 5. missing         (140 times)
 6. features        (140 times)


In [7]:
print(" TOP 10 HIGHEST VALUE CHURNED CUSTOMERS")
print(pd.read_sql("""
    SELECT account_id, plan_tier, mrr_amount as monthly_revenue, tenure_months, ticket_count
    FROM master 
    WHERE churn_flag = 1 
    ORDER BY mrr_amount DESC 
    LIMIT 10
""", conn))

 TOP 10 HIGHEST VALUE CHURNED CUSTOMERS
  account_id   plan_tier  monthly_revenue  tenure_months  ticket_count
0   A-118f1c  Enterprise            25472            0.1             1
1   A-d4e0d4  Enterprise            23283            0.1             4
2   A-ce550d  Enterprise            21691            0.0             7
3   A-a6d261  Enterprise            20497            7.0             4
4   A-c37cab  Enterprise            19502            2.3             4
5   A-c37cab  Enterprise            19502            2.3             4
6   A-66224b  Enterprise            18308            0.3             6
7   A-5c9849  Enterprise            18109            0.5             4
8   A-eb7c38  Enterprise            17910            4.9             5
9   A-f44180  Enterprise            17711            7.0             2


In [8]:
pd.read_sql("SELECT * FROM master", conn).to_csv("master_for_tableau.csv", index=False)
print(" Exported master_for_tableau.csv for Tableau / Power BI!")

 Exported master_for_tableau.csv for Tableau / Power BI!
